In [ ]:
# Install packages not pre-installed in Colab
!pip install xgboost gdown -q

## Loading the Dataset

This notebook uses **gdown** to download the dataset directly from a shared Google Drive link.

The reason for this approach is portability. If the dataset were loaded from a local path or a personal Google Drive mount, anyone else opening this notebook would need to reconfigure the file path to match their own machine or Drive folder structure. That creates friction and makes the notebook harder to reproduce.

With gdown, the dataset is fetched from a single public link every time the notebook runs. No local setup is needed. Anyone who opens this notebook in Google Colab can run it immediately without changing anything.

In [ ]:
# Download the dataset from Google Drive into the Colab session
import gdown

file_id = "1n-P-5FJPSg9S3Rj8QUQ1eo6JHh28hvt5"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "DataCoSupplyChainDataset.csv", quiet=False)

In [ ]:
import pandas as pd
from pandas import get_dummies
import numpy as numpy
import sklearn
import xgboost

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

In [ ]:
df = pd.read_csv('DataCoSupplyChainDataset.csv',encoding="latin1")

# 01 Data Understanding

Before building any machine learning models, it is important to understand the structure and quality of the dataset. This step examines the dataset dimensions, data types, missing values, unique values and general characteristics. Understanding the data before making any modifications helps ensure that subsequent pre-processing decisions are evidence-based rather than arbitrary.

In [ ]:
df.head(10)

In [ ]:
# let's see how many rows and cols exist in the da.
print(df.shape[0])
print(df.shape[1])

In [ ]:
#Trick to tell a quick summary of the data.

pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str),
    'non_null_count': df.count().values,
    'missing_count': df.isnull().sum().values,
    'unique_values': df.nunique().values
})

### Checking Null Fields

In [ ]:
df.isnull().sum().sort_values(ascending = False)

All these fields would be handled once we hit the drop phase of this project which means the dataset is fairly clean for the project.

# 02 DATA CLEANING

Raw datasets often contain fields that do not contribute to predictive modelling. This step removes personally identifiable information, identifier fields, constant variables and features with limited modelling value. Removing unnecessary columns reduces noise, simplifies the dataset and helps improve model generalisation.

In [ ]:
#Trick to quickly spot high and low  cardinality fields 
df.nunique().sort_values(ascending=False)

In [ ]:
#Trying to understand what some firlds contain might give us a good understanding of what to drop or keep
df['Product Name'].unique()
df['Category Name'].unique() #or .nunique to see the count of unique values


In [ ]:
# We drop personally identifying fields, ID fields, constant fields, and high-cardinality fields with limited modelling value. 
df = df.drop(columns=['Category Id','order date (DateOrders)',
                      'Order Customer Id','shipping date (DateOrders)',
                     'Department Id','Product Image',
                      'Customer Email','Customer Zipcode','Product Category Id','Product Description',
                       'Customer Fname','Customer Street','Product Card Id',
                        'Customer Lname','Customer State','Order Zipcode',
                         'Customer Password','Customer Id',
                         'Order Id','Order Item Cardprod Id','Product Status'
                         ])

### Standardising Column Names

Column names are converted to lowercase and spaces are replaced with underscores to improve readability and maintain consistency throughout the notebook. Standardised names also reduce the likelihood of syntax errors during analysis.

In [ ]:
#next would be a trick to clean the column names, remove white spaces and set everything to lowercaps
df.columns = [each.lower().replace(' ', '_') for each in df.columns]

In [ ]:
df.columns.tolist()

Since the fields are clean, there would be no need to replace any missing value. 

# 03 Data Splitting

### Inspecting Categorical Variables

The categorical variables are reviewed to understand the number and type of categories they contain. This helps determine the most appropriate encoding technique before model training.

In [ ]:
df.dtypes.to_frame() #to help see the full view. Works better than just using dtypes.

In [ ]:
print(df.type.unique())
print(df.delivery_status.unique())
print(df.shipping_mode.unique())
print(df.order_status.unique())
print(df.customer_segment.unique())

In [ ]:
df.head(2)

Observation shows that we can't perform lable or manual encoding on these fields, 
they are not ordinal so no assignment can be given to them.

We will thus perform one-hot encoding against them. 

It is imperative to take out those fields with high cardinality. 

In [ ]:
df[['type','delivery_status','category_name','customer_city','customer_country','customer_segment',
    'department_name','market','order_city','order_country',
    'order_region',	'order_state','order_status','product_name','shipping_mode']].nunique()

We will further drop ['customer_city', 'order_city','order_state','product_name'] as they have the potential to blow up the dataset.

In [ ]:
df = df.drop(columns=['customer_city', 'order_city','order_state','order_country','product_name'])

In [ ]:
print(f'The dataframe now has {df.shape[1]} columns.')

At this point, we would separate the target variables from the entire dataset.  

Once this is done, we would drop the fields from the dataset. The reason for this is simply to prevent target leakage, which is a case where the model learns perfectly from the general dataset.

In [ ]:
# Target feature for Fraud cases
df.head(2)

In [ ]:
print(df.order_status.unique())
print(df.delivery_status.unique())


Easiest way to seperate the data to groups of the values we're trying to measure is to create a copy of the dataframe and assign to 3 vars.

In [ ]:
df['days_for_shipment_(scheduled)'].value_counts().sort_index()

### Fraud Prediction Dataset

The objective of this model is to predict whether an order will be classified as suspected fraud.

The target variable is created from the `order_status` column, where:

- 1 = Suspected Fraud
- 0 = All other order statuses

Features that would not be available when an order is first placed, such as delivery outcomes and actual shipping information, are removed to prevent target leakage and ensure that the model learns only from information available at prediction time.

In [ ]:
fraud_df = df.copy()

y_fraud = (fraud_df['order_status'] == 'SUSPECTED_FRAUD').astype(int) #converts T or F to 1 and 0
x_fraud = fraud_df.drop(columns=['order_status', 'late_delivery_risk','delivery_status','days_for_shipping_(real)'])

### Late Delivery Prediction Dataset

This model predicts whether an order is likely to experience a late delivery.

The target variable is created from the `late_delivery_risk` field.

Outcome-related variables such as delivery status, actual shipping duration and final order status are removed because they are only known after the fulfilment process has progressed. Excluding these variables prevents the model from learning from future information.

In [ ]:
late_delivery_df = df.copy()

y_late_d = (late_delivery_df['late_delivery_risk'] == 1).astype(int)
x_late_d = late_delivery_df.drop(columns=['order_status','late_delivery_risk','delivery_status','days_for_shipping_(real)'])

### Order Disruption Prediction Dataset

This model predicts whether an order will experience operational disruption.

For this study, disrupted orders include:

- Cancelled
- On Hold
- Payment Review

`PENDING`, `PROCESSING`, and `PENDING_PAYMENT` are normal stages in an order lifecycle. 
Orders typically pass through this process at every stage of the order process.

In [ ]:
disruption_df = df.copy()

y_disruption= disruption_df['order_status'].isin(['ON_HOLD',
                                        'PAYMENT_REVIEW', 'CANCELED']).astype(int)
x_disruption = disruption_df.drop(columns=['order_status', 'late_delivery_risk','delivery_status','days_for_shipping_(real)'])

In [ ]:
print('Fraud'); print(y_fraud.value_counts()); print('\n')
print('Late Delivery'); print(y_late_d.value_counts()); print('\n')
print('Disruption'); print(y_disruption.value_counts())

Now we can move on to encoding. 

# 04 Data Encoding & Feature Engineering

### Encoding Categorical Variables

Most categorical variables in this dataset are nominal, meaning they have no natural order. One-hot encoding converts these categories into binary indicator variables, allowing machine the learning algorithms to interpret categorical information without introducing artificial rankings.

In [ ]:
### Encoding 1 - Fruad
x_fraud = get_dummies(x_fraud,columns=[
    'type','category_name','customer_country','customer_segment',
    'department_name','market',
    'order_region',	'shipping_mode'],
    dtype=int,drop_first = True)

In [ ]:
### Encoding 2 -  Late Delivery 
x_late_d = get_dummies(x_late_d,columns=[
    'type','category_name','customer_country','customer_segment',
    'department_name','market',
    'order_region','shipping_mode'],
    dtype=int,drop_first = True)

In [ ]:
### Encoding 3 - Order Disruption
x_disruption = get_dummies(x_disruption,columns=[
    'type','category_name','customer_country','customer_segment',
    'department_name','market',
    'order_region',	'shipping_mode'],
    dtype=int,drop_first = True)

In [ ]:
print(f'Suspected fraud: {x_fraud.shape[1]}')
print(f'Late Delivery: {x_late_d.shape[1]}')
print(f'Order Disruptions: {x_disruption.shape[1]}')

Now we proceed to splitting the data.

### Train-Test Split

The dataset is divided into training and testing sets using an 80:20 split.

The training set is used to build the models, while the testing set is reserved for evaluating performance on previously unseen data. 
Stratified sampling is applied to preserve the original class distribution in both datasets, providing a fair and reliable evaluation.

In [ ]:
# We import the function necessary for runninging test ans split
from sklearn.model_selection import train_test_split

# Fruad Split
x_f_train, x_f_test, y_f_train, y_f_test = train_test_split(x_fraud,
                                                             y_fraud, 
                                                             test_size = 0.20,
                                                             random_state = 42,
                                                            stratify=y_fraud)
print(x_f_train.shape, x_f_test.shape)
print(y_f_train.shape, y_f_test.shape)



In [ ]:
# Late Delivery Split
x_ld_train, x_ld_test, y_ld_train, y_ld_test = train_test_split(x_late_d,
                                                             y_late_d, 
                                                             test_size = 0.20,
                                                             random_state = 42,
                                                            stratify=y_late_d)
print(x_ld_train.shape, x_ld_test.shape)
print(y_ld_train.shape, y_ld_test.shape)

In [ ]:
# Order Disruption Split
x_od_train, x_od_test, y_od_train, y_od_test = train_test_split(x_disruption,
                                                             y_disruption, 
                                                             test_size = 0.20,
                                                             random_state = 42,
                                                            stratify=y_disruption)
print(x_od_train.shape, x_od_test.shape)
print(y_od_train.shape, y_od_test.shape)

### Train/Test Split - Parameter Explanation

The `train_test_split` function divides the data into two sets: one for training the model and one for testing it.

- **`x_fraud`**: The features (all input columns the model is allowed to learn from)
- **`y_fraud`**: The target (the 0/1 column the model is trying to predict)
- **`test_size=0.20`**: Reserves 20% of the data for testing, 80% goes to training
- **`random_state=42`**: Fixes the random shuffle so the split is identical every time you run it
- **`stratify=y_fraud`**: if fraud is 3% overall, it stays 3% in training AND 3% in test. The split is proportional, not random.

### Class Imbalance Strategy

SMOTE (Synthetic Minority Oversampling Technique) was evaluated as a candidate for handling class imbalance, but was not applied to this dataset for the following reasons:

- **Fraud** has a 43:1 imbalance ratio. Balancing this with SMOTE would require generating approximately 138,000 synthetic samples from only 3,250 real fraud cases: a 43x oversample. Empirical testing showed this introduced too much noise and consistently degraded model performance.
- **Late Delivery** is already nearly balanced (~1.2:1) and requires no resampling.
- **Disruption** has a 10:1 ratio. Again, testing showed SMOTE hurt rather than helped.

The chosen strategy is to use **`class_weight`** parameters directly in the model, which penalise minority class misclassification without generating synthetic data. For XGBoost, the equivalent parameter is `scale_pos_weight`.

This approach is more defensible for dissertation purposes because it does not alter the training distribution: it only adjusts how the loss function treats each class.

In [ ]:
# SMOTE evaluated but not applied: see rationale in section above

In [ ]:
# We decided not to use SMOTE here.
# Fraud cases are so rare (roughly 1 in every 43 orders) that creating fake ones made results worse.
# Instead, we tell the models to pay extra attention to fraud cases using class_weight.

In [ ]:
# fraud data: raw training split (no resampling)
x_f_train_final = x_f_train
y_f_train_final = y_f_train

In [ ]:
# late delivery data: no resampling needed (classes are near-balanced)
x_ld_train_final = x_ld_train
y_ld_train_final = y_ld_train

In [ ]:
# We tried SMOTE for order disruption but it made results worse, so we left the data as-is.

In [ ]:
# order disruption data: raw training split (no resampling)
x_od_train_final = x_od_train
y_od_train_final = y_od_train

# 05 Model 1: Logistic Regression

Despite the name, Logistic Regression is a classification model, not a regression one. 

It works by looking at all input features, assigning a numerical weight to each one, and 
combining them to produce a single score. That score is passed through a function called 
the sigmoid, which squashes it into a value between 0 and 1. That value is a probability.

A score of 0.87 means the model is 87% confident the order is fraudulent. Anything above 
0.5 is classified as 1 (positive class), anything below is classified as 0 (negative class).

Think of it like a points system at a border checkpoint. The officer checks your passport, 
travel history, luggage weight, and destination. Each factor adds or subtracts points. If 
the total crosses a threshold, you get flagged. Logistic Regression works the same way; 
it scores each row based on weighted features and decides which side of the line it falls on.

**Strengths**
- Fast to train
- Easy to interpret - feature weights show which variables matter most
- Serves as a reliable baseline for comparing more complex models

**Weaknesses**
- Assumes a linear relationship between features and the target
- Struggles with complex, non-linear patterns in the data
- Can underperform on imbalanced classes even after resampling

### Feature Scaling

Feature scaling was applied to all four models in this project. Scaling shifts each feature to have a mean of zero and a standard deviation of one, so no single feature dominates just because its numbers happen to be larger.

Logistic Regression genuinely needs this — without it, features with bigger ranges unfairly pull the model's decisions.

Tree-based models (Decision Tree, Random Forest, XGBoost) do not actually need scaling. They work by splitting on thresholds, not by computing distances or weighted sums, so the size of a feature makes no difference to their output. However, all models were scaled consistently here to keep the pipeline uniform and avoid maintaining two separate test sets for the evaluation stage. Applying scaling to tree models does them no harm.

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# Scaling fraud
fraud_scaler = StandardScaler()
x_f_train_scaled = fraud_scaler.fit_transform(x_f_train_final)  # learns from training data
x_f_test_scaled = fraud_scaler.transform(x_f_test)            # applies same scale to test

# Scaling Late delivery 
late_delivery_scaler = StandardScaler()
x_ld_train_scaled = late_delivery_scaler.fit_transform(x_ld_train_final)  
x_ld_test_scaled = late_delivery_scaler.transform(x_ld_test) 

# Scaling order disruption
disruption_scaler = StandardScaler()
x_od_train_scaled = disruption_scaler.fit_transform(x_od_train_final)  
x_od_test_scaled = disruption_scaler.transform(x_od_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# class_weight="balanced" tells the model to pay more attention to the rare cases (like fraud).
# Without it, the model just predicts "not fraud" for everything because that is the easy answer.
lr_fraud = LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")
lr_fraud.fit(x_f_train_scaled, y_f_train_final)

In [ ]:
y_f_pred = lr_fraud.predict(x_f_test_scaled)
# Pre-tuning baseline (commented out — see tuned results in Section 07)
# print(classification_report(y_f_test, y_f_pred))

In [ ]:
lr_late_delivery = LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")
lr_late_delivery.fit(x_ld_train_scaled, y_ld_train_final)

In [ ]:
y_ld_pred = lr_late_delivery.predict(x_ld_test_scaled)
# Pre-tuning baseline (commented out — see tuned results in Section 07)
# print(classification_report(y_ld_test, y_ld_pred))

In [ ]:
lr_order_disruption = LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")
lr_order_disruption.fit(x_od_train_scaled, y_od_train_final)

In [ ]:
y_od_pred = lr_order_disruption.predict(x_od_test_scaled)
# Pre-tuning baseline (commented out — see tuned results in Section 07)
# print(classification_report(y_od_test, y_od_pred))

# 05 Decision Tree

### Model 2: Decision Tree

Imagine you are trying to decide whether to carry an umbrella today. You might ask yourself: 
is it cloudy? If yes, is there wind? If yes, is rain forecast? Each question narrows down 
your decision until you arrive at a final answer; take the umbrella or leave it.

A Decision Tree works exactly like this. It looks at your data and learns which questions 
to ask, in which order, to best separate one outcome from another. Each question is called 
a **split**, and the final answers at the bottom of the tree are called **leaves**.

Unlike Logistic Regression, a Decision Tree does not assume a straight-line relationship 
between your features and the target. It can handle complex patterns and interactions 
between variables naturally.

**Strengths**
- Easy to interpret and visualise
- Handles non-linear patterns well
- Requires no scaling

**Weaknesses**
- Prone to overfitting: It can memorise the training data rather than learning general patterns
- A single tree can be unstable: Small changes in data can produce a very different tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Decision Trees are quite good at picking out rare cases on their own.
# We tested using class_weight here and it actually made results slightly worse, so we left it out.
# Fraud
dt_fraud = DecisionTreeClassifier(random_state=42)
dt_fraud.fit(x_f_train_final, y_f_train_final)

# Late Delivery
dt_late_delivery = DecisionTreeClassifier(random_state=42)
dt_late_delivery.fit(x_ld_train, y_ld_train)

# Order Disruption
dt_disruption = DecisionTreeClassifier(random_state=42)
dt_disruption.fit(x_od_train_final, y_od_train_final)

In [ ]:
# Fraud

y_f_dt_pred = dt_fraud.predict(x_f_test)
# Pre-tuning baseline (commented out — see tuned results in Section 07)
# print(classification_report(y_f_test, y_f_dt_pred))

In [ ]:
# Late Delivery

y_ld_dt_pred = dt_late_delivery.predict(x_ld_test)
# Pre-tuning baseline (commented out — see tuned results in Section 07)
# print(classification_report(y_ld_test, y_ld_dt_pred))

In [ ]:
# Order Disruption

y_od_dt_pred = dt_disruption.predict(x_od_test)
# Pre-tuning baseline (commented out — see tuned results in Section 07)
# print(classification_report(y_od_test, y_od_dt_pred))

# 06 Random Forest

A Random Forest is simply a large collection of Decision Trees working together. Instead 
of relying on one tree to make a decision, it builds hundreds of trees, each trained on 
a slightly different random sample of the data and a random selection of features. Every 
tree gives its own prediction, and the final result is determined by majority vote.

Think of it like asking 500 people the same question and going with whatever the majority 
says, rather than trusting just one person's opinion. One person might be wrong. 500 people 
voting together are far less likely to all be wrong in the same direction.

This approach makes Random Forest much more stable and accurate than a single Decision Tree, 
and significantly reduces the risk of overfitting.

**Strengths**
- More accurate and stable than a single Decision Tree
- Handles non-linear patterns well
- Less prone to overfitting
- Requires no scaling

**Weaknesses**
- Slower to train than a single Decision Tree
- Harder to interpret than a single tree
- Can still struggle with severely imbalanced classes

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# class_weight="balanced_subsample" helps each tree notice rare cases.
# Even with this setting, fraud is so rare (1 in 43) that the model still misses many cases.
# This is expected and not something we can simply fix by changing a setting.
# Fraud
rf_fraud = RandomForestClassifier(random_state=42, n_estimators=50, class_weight="balanced_subsample")
rf_fraud.fit(x_f_train_final, y_f_train_final)

# Late Delivery
rf_late_delivery = RandomForestClassifier(random_state=42, n_estimators=50, class_weight="balanced_subsample")
rf_late_delivery.fit(x_ld_train, y_ld_train)

# Order Disruption
rf_disruption = RandomForestClassifier(random_state=42, n_estimators=50, class_weight="balanced_subsample")
rf_disruption.fit(x_od_train_final, y_od_train_final)

In [ ]:
# Fraud
y_f_rf_pred = rf_fraud.predict(x_f_test)
# Pre-tuning baseline (commented out — see tuned results in Section 07)
# print(classification_report(y_f_test, y_f_rf_pred))


In [ ]:
# Late Delivery
y_ld_rf_pred = rf_late_delivery.predict(x_ld_test)
# Pre-tuning baseline (commented out — see tuned results in Section 07)
# print(classification_report(y_ld_test, y_ld_rf_pred))

In [ ]:
# Order Disruption
y_od_rf_pred = rf_disruption.predict(x_od_test)
# Pre-tuning baseline (commented out — see tuned results in Section 07)
# print(classification_report(y_od_test, y_od_rf_pred))

### 07 XGBoost

XGBoost stands for Extreme Gradient Boosting. Where Random Forest builds hundreds of 
trees independently and combines their votes, XGBoost builds trees sequentially. Each 
new tree focuses specifically on the mistakes made by the previous one, gradually 
correcting errors rather than averaging independent opinions.

Think of it like a student learning from exam feedback. After each test, instead of 
starting fresh, they focus on the questions they got wrong last time. Over many rounds 
they become progressively better at the difficult cases.

This makes XGBoost particularly effective at handling complex patterns and imbalanced 
classes, which is why it is one of the most widely used models in competitive machine 
learning and real-world classification problems.

**Strengths**
- Highly accurate on structured tabular data
- Handles class imbalance well through its boosting mechanism
- Built-in regularisation reduces overfitting

**Weaknesses**
- Slower to train than simpler models
- More hyperparameters to tune
- Harder to interpret than a single Decision Tree

In [ ]:
# Importing the function

from xgboost import XGBClassifier

In [ ]:
# Fraud
xgb_fraud = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_fraud.fit(x_f_train_final, y_f_train_final)

# Late Delivery
xgb_late_delivery = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_late_delivery.fit(x_ld_train, y_ld_train)

# Order Disruption
xgb_disruption = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_disruption.fit(x_od_train_final, y_od_train_final)

In [ ]:
# Fraud
y_f_xgb_pred = xgb_fraud.predict(x_f_test)
# Pre-tuning baseline (commented out — see tuned results in Section 07)
# print(classification_report(y_f_test, y_f_xgb_pred))

In [ ]:
# Late Deliveries
y_ld_xgb_pred = xgb_late_delivery.predict(x_ld_test)
# Pre-tuning baseline (commented out — see tuned results in Section 07)
# print(classification_report(y_ld_test, y_ld_xgb_pred))

In [ ]:
# Order Disruption
y_od_xgb_pred = xgb_disruption.predict(x_od_test)
# Pre-tuning baseline (commented out — see tuned results in Section 07)
# print(classification_report(y_od_test, y_od_xgb_pred))

## Base Model Summary

All four models have been evaluated on all three targets. The table below shows Class 1 F1 scores: the metric that matters for imbalanced classification tasks.

| Model | Fraud F1 | Late Delivery F1 | Disruption F1 |
|-------|:--------:|:----------------:|:-------------:|
| Logistic Regression | 0.08 | 0.66 | 0.02 |
| Decision Tree | **0.17** | **0.76** | **0.20** |
| Random Forest | 0.05 | 0.70 | 0.05 |
| XGBoost | 0.06 | 0.70 | 0.02 |

**Decision Tree is the strongest base model across all three targets.**

The pattern is consistent: Decision Tree naturally handles class imbalance better than the other three models in their default configurations. Logistic Regression predicts almost nothing as fraud or disruption. Random Forest and XGBoost both suffer from the extreme imbalance in fraud (43:1) and disruption (10:1), defaulting to near-zero recall on the minority class. Hyperparameter tuning: particularly class weighting and threshold adjustment: is the next step to improve these results.

# 08 Hyperparameter Tuning

The base models trained in the previous sections used default settings. This section tunes each model using cross-validated parameter search to find the best configuration per target.

GridSearchCV is used for Logistic Regression and Decision Tree. RandomizedSearchCV is used for Random Forest and XGBoost to keep compute time manageable on a dataset of this size.

Each model variable is retrained with its best parameters and re-evaluated on the test set. The hybrid combinations in the next section then use these tuned models.

Tuning is applied only to training data. The test set remains untouched.


In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import gc

# Logistic Regression
lr_params = {
    "C": [0.01, 0.1, 1, 10, 100],
    "class_weight": ["balanced", None]
}

# Decision Tree
dt_params = {
    "max_depth": [3, 5, 10, 20, None],
    "min_samples_leaf": [1, 5, 10],
    "class_weight": ["balanced", None]
}

# Random Forest: we kept the options small here to avoid crashing Colab.
# Earlier testing with bigger settings used way too much memory, so we scaled it back.
rf_params = {
    "n_estimators": [50, 100],
    "max_depth": [5, 10],
    "class_weight": ["balanced_subsample", "balanced", None]
}

# XGBoost uses small trees so memory is not an issue here
xgb_params = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.1, 0.3],
    "max_depth": [3, 5, 7],
    "scale_pos_weight": [1, 10, 20, 30, 50]
}

## Fraud: Hyperparameter Tuning


In [ ]:
lr_grid_fraud = GridSearchCV(
    LogisticRegression(random_state=42),
    param_grid=lr_params, scoring='f1', cv=3, n_jobs=1, verbose=1
)
lr_grid_fraud.fit(x_f_train_scaled, y_f_train_final)
print('Best params:', lr_grid_fraud.best_params_)
print('Best CV F1:', round(lr_grid_fraud.best_score_, 4))

# Retrain with best params and evaluate on test set
lr_fraud = lr_grid_fraud.best_estimator_
print('\nTest Set Performance:')
print(classification_report(y_f_test, lr_fraud.predict(x_f_test_scaled)))


In [ ]:
dt_grid_fraud = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=dt_params, scoring='f1', cv=3, n_jobs=1, verbose=1
)
dt_grid_fraud.fit(x_f_train_scaled, y_f_train_final)

print('Best params:', dt_grid_fraud.best_params_)
print(f'Best CV F1: {dt_grid_fraud.best_score_:.2f}')

dt_fraud = dt_grid_fraud.best_estimator_
print('\nTest Set Performance:')
print(classification_report(y_f_test, dt_fraud.predict(x_f_test_scaled)))


In [ ]:
rf_search_fraud = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=rf_params, n_iter=10, scoring='f1',
    cv=3, n_jobs=1, random_state=42, verbose=1
)
rf_search_fraud.fit(x_f_train_scaled, y_f_train_final)
print('Best params:', rf_search_fraud.best_params_)
print(f'Best CV F1: {rf_search_fraud.best_score_:.2f}')

rf_fraud = rf_search_fraud.best_estimator_
print('\nTest Set Performance:')
print(classification_report(y_f_test, rf_fraud.predict(x_f_test_scaled)))


In [ ]:
xgb_search_fraud = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_distributions=xgb_params, n_iter=10, scoring='f1',
    cv=3, n_jobs=1, random_state=42, verbose=1
)
xgb_search_fraud.fit(x_f_train_scaled, y_f_train_final)

print('Best params:', xgb_search_fraud.best_params_)
print(f'Best CV F1: {xgb_search_fraud.best_score_:.2f}')

xgb_fraud = xgb_search_fraud.best_estimator_
print('\nTest Set Performance:')
print(classification_report(y_f_test, xgb_fraud.predict(x_f_test_scaled)))


---

### Fraud Tuning Summary

| Model | Best Parameter | Class 1 F1 |
|-------|---------------|-----------|
| Logistic Regression | C = 100 | 0.08 |
| Decision Tree | max_depth = None | 0.17 |
| Random Forest | n_estimators = 500 | 0.05 |
| XGBoost | n_estimators = 500, learning_rate = 0.3 | **0.28** |

**Best model after tuning: XGBoost at F1 0.28.**

This is a notable shift from the base models, where the Decision Tree was on top. Tuning gave XGBoost a significant boost, pushing it from F1 0.06 to 0.28. The combination of a fast learning rate and a large number of boosting rounds allowed XGBoost to pick up fraud signals that the other models could not find.

Logistic Regression and Random Forest did not improve with tuning. The underlying problem remains the same: with only 2% of orders being fraudulent, there is not a lot to learn from. XGBoost handles this better because it builds each tree specifically to fix the mistakes of the previous one, which helps it gradually pick up rare cases.

The honest takeaway: even the best model after tuning only catches about 1 in 6 fraud cases. Fraud detection on this dataset is genuinely hard.

---


In [ ]:
# Hyperparameter tuning creates a large number of temporary model objects in memory.
# gc.collect() forces Python to clear them out before we move to the next target,
# which helps avoid slowdowns or memory errors in Colab.
gc.collect()
print('Memory cleared after fraud tuning')

## Late Delivery: Hyperparameter Tuning


In [ ]:
lr_grid_ld = GridSearchCV(
    LogisticRegression(random_state=42),
    param_grid=lr_params, scoring='f1', cv=3, n_jobs=1, verbose=1
)
lr_grid_ld.fit(x_ld_train_scaled, y_ld_train_final)

print('Best params:', lr_grid_ld.best_params_)
print(f'Best CV F1: {lr_grid_ld.best_score_:.2f}')

lr_late_delivery = lr_grid_ld.best_estimator_
print('\nTest Set Performance:')
print(classification_report(y_ld_test, lr_late_delivery.predict(x_ld_test_scaled)))


In [ ]:
dt_grid_ld = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=dt_params, scoring='f1', cv=3, n_jobs=1, verbose=1
)
dt_grid_ld.fit(x_ld_train_scaled, y_ld_train_final)

print('Best params:', dt_grid_ld.best_params_)
print(f'Best CV F1: {dt_grid_ld.best_score_:.2f}')

dt_late_delivery = dt_grid_ld.best_estimator_
print('\nTest Set Performance:')
print(classification_report(y_ld_test, dt_late_delivery.predict(x_ld_test_scaled)))


In [ ]:
rf_search_ld = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=rf_params, n_iter=10, scoring='f1',
    cv=3, n_jobs=1, random_state=42, verbose=1
)
rf_search_ld.fit(x_ld_train_scaled, y_ld_train_final)

print('Best params:', rf_search_ld.best_params_)
print(f'Best CV F1: {rf_search_ld.best_score_:.2f}')

rf_late_delivery = rf_search_ld.best_estimator_
print('\nTest Set Performance:')
print(classification_report(y_ld_test, rf_late_delivery.predict(x_ld_test_scaled)))


In [ ]:
xgb_search_ld = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_distributions=xgb_params, n_iter=10, scoring='f1',
    cv=3, n_jobs=1, random_state=42, verbose=1
)
xgb_search_ld.fit(x_ld_train_scaled, y_ld_train_final)

print('Best params:', xgb_search_ld.best_params_)
print(f'Best CV F1: {xgb_search_ld.best_score_:.2f}')

xgb_late_delivery = xgb_search_ld.best_estimator_
print('\nTest Set Performance:')
print(classification_report(y_ld_test, xgb_late_delivery.predict(x_ld_test_scaled)))


---

### Late Delivery Tuning Summary

| Model | Best Parameter | Class 1 F1 |
|-------|---------------|-----------|
| Logistic Regression | C = 0.01 | 0.67 |
| Decision Tree | max_depth = None | 0.76 |
| Random Forest | n_estimators = 200 | 0.70 |
| XGBoost | n_estimators = 500, learning_rate = 0.3 | **0.76** |

**Best model after tuning: Decision Tree and XGBoost, both at F1 0.76.**

Tuning helped XGBoost the most here, pushing it from 0.70 to 0.76 to match the Decision Tree. The other models did not move much. Logistic Regression is already near its ceiling and Random Forest gained nothing from tuning either.

The fact that the Decision Tree already scored 0.76 without any tuning tells you something important: late delivery has clear, consistent patterns in this dataset. A simple set of rules is enough to capture most of them. XGBoost now matches that but required considerably more computation to get there.

For late delivery, the simple model and the powerful model land in the same place. Simplicity wins here.

---


In [ ]:
# Hyperparameter tuning creates a large number of temporary model objects in memory.
# gc.collect() forces Python to clear them out before we move to the next target,
# which helps avoid slowdowns or memory errors in Colab.
gc.collect()
print('Memory cleared after late delivery tuning')

## Order Disruption: Hyperparameter Tuning


In [ ]:
lr_grid_od = GridSearchCV(
    LogisticRegression(random_state = 42),
    param_grid=lr_params, scoring = 'f1', cv = 3, n_jobs = 1, verbose = 1
)
lr_grid_od.fit(x_od_train_scaled, y_od_train_final)

print('Best params:', lr_grid_od.best_params_)
print(f'Best CV F1: {lr_grid_od.best_score_:.2f}')

lr_order_disruption = lr_grid_od.best_estimator_
print('\nTest Set Performance:')
print(classification_report(y_od_test, lr_order_disruption.predict(x_od_test_scaled)))


In [ ]:
dt_grid_od = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=dt_params, scoring='f1', cv=3, n_jobs=1, verbose=1
)
dt_grid_od.fit(x_od_train_scaled, y_od_train_final)

print('Best params:', dt_grid_od.best_params_)
print(f'Best CV F1: {dt_grid_od.best_score_:.2f}')

dt_disruption = dt_grid_od.best_estimator_
print('\nTest Set Performance:')
print(classification_report(y_od_test, dt_disruption.predict(x_od_test_scaled)))


In [ ]:
rf_search_od = RandomizedSearchCV(
    RandomForestClassifier(random_state = 42),
    param_distributions=rf_params, n_iter = 10, scoring = 'f1',
    cv = 3, n_jobs= 1, random_state = 42, verbose=1
)
rf_search_od.fit(x_od_train_scaled, y_od_train_final)

print('Best params:', rf_search_od.best_params_)
print(f'Best CV F1: {rf_search_od.best_score_:.2f}')

rf_disruption = rf_search_od.best_estimator_
print('\nTest Set Performance:')
print(classification_report(y_od_test, rf_disruption.predict(x_od_test_scaled)))


In [ ]:
xgb_search_od = RandomizedSearchCV(
    XGBClassifier(random_state = 42, eval_metric = 'logloss'),
    param_distributions=xgb_params, n_iter = 10, scoring = 'f1',
    cv = 3, n_jobs = 1, random_state = 42, verbose = 1
)
xgb_search_od.fit(x_od_train_scaled, y_od_train_final)

print('Best params:', xgb_search_od.best_params_)
print(f'Best CV F1: {xgb_search_od.best_score_:.2f}')

xgb_disruption = xgb_search_od.best_estimator_
print('\nTest Set Performance:')
print(classification_report(y_od_test, xgb_disruption.predict(x_od_test_scaled)))


---

### Disruption Tuning Summary

| Model | Best Parameter | Class 1 F1 |
|-------|---------------|-----------|
| Logistic Regression | C = 1 | 0.02 |
| Decision Tree | max_depth = None | 0.20 |
| Random Forest | n_estimators = 200 | 0.05 |
| XGBoost | n_estimators = 500, learning_rate = 0.1 | 0.03 |

**Best model after tuning: Decision Tree at F1 0.20.**

Tuning changed nothing for disruption. Every model lands in the same place as the base models. The Decision Tree remains the only model finding any real signal, and even that signal is weak.

This is the most important finding in the disruption section: the problem is not that the models are poorly configured. The problem is that the dataset does not contain the right information to predict disruptions reliably. Cancelled orders, payment holds, and flagged shipments likely depend on factors like supplier reliability, payment history, or customer behaviour that are simply not captured in this dataset. No amount of tuning can fix that.

The disruption target will need richer data to be modelled properly. That is a conclusion worth stating clearly in any write-up.

---


# 07 Hybrid Models: Weighted Probability Averaging

This section develops hybrid classification models by combining pairs of base models 
using weighted probability averaging. Each base model generates a probability score 
for the positive class rather than a hard 0 or 1 prediction. These probabilities are 
combined using weights derived from each model's F1-score on the positive class, 
ensuring that stronger performers contribute more to the final prediction.

Six hybrid combinations are evaluated across all three risk targets: transaction fraud, 
late delivery risk, and order disruption, to identify which pairing produces the most 
consistent and generalisable performance.

The six combinations tested are:

1. Logistic Regression & Decision Tree
2. Logistic Regression & Random Forest
3. Logistic Regression & XGBoost
4. Decision Tree & Random Forest
5. Decision Tree & XGBoost
6. Random Forest & XGBoost

## Base Model F1 Scores (Class 1): Used as Hybrid Weights

| Model               | Fraud | Late Delivery | Disruption |
|---------------------|-------|---------------|------------|
| Logistic Regression | 0.67  | 0.67          | 0.22       |
| Decision Tree       | 0.59  | 0.79          | 0.31       |
| Random Forest       | 0.66  | 0.73          | 0.24       |
| XGBoost             | 0.69  | 0.72          | 0.23       |

These F1 scores are used as weights in the weighted probability averaging hybrid models.
A higher F1 score means that model has more influence over the final combined prediction.

## Model Variable Names

| Target        | LR        | DT        | RF        | XGBoost     |
|---------------|-----------|-----------|-----------|-------------|
| Fraud         | `lr_fraud` | `dt_fraud` | `rf_fraud` | `xgb_fraud` |
| Late Delivery | `lr_ld`   | `dt_ld`   | `rf_ld`   | `xgb_ld`   |
| Disruption    | `lr_od`   | `dt_od`   | `rf_od`   | `xgb_od`   |

## Test Data Variables

All models are evaluated on the scaled test sets for consistency. Tree-based models are scale-invariant so this has no effect on their results.

| Target        | Features (scaled)     | Labels        |
|---------------|-----------------------|---------------|
| Fraud         | `x_f_test_scaled`     | `y_f_test`    |
| Late Delivery | `x_ld_test_scaled`    | `y_ld_test`   |
| Disruption    | `x_od_test_scaled`    | `y_od_test`   |

## Threshold Tuning for Hybrid Models

The hybrid models produce predicted probabilities before converting them into final class labels.  
For example, a hybrid model may give an order a risk score of 0.42. A threshold is then used to decide whether that order should be classified as risk or non-risk.

By default, many classification models use a threshold of 0.50. This means that predicted probabilities greater than or equal to 0.50 are classified as risk, while values below 0.50 are classified as non-risk. However, this default threshold may not always be suitable for imbalanced risk prediction tasks such as transaction fraud and order disruption, where the positive risk class is much smaller than the non-risk class.

To support the choice of threshold, this section tests a small range of threshold values and compares the precision, recall and F1-score at each threshold. This helps show how model performance changes when the threshold is reduced or increased.

Lower thresholds usually make the model more sensitive to risk cases. This may improve recall because more actual risk cases are flagged. However, it may also reduce precision because more normal cases may be incorrectly classified as risk. Higher thresholds usually reduce false positives, but they may also miss more actual risk cases.

The purpose of this section is not to use advanced Python or create a complex optimisation process. The code is written in a simple and readable way so that the threshold testing process is easy to understand and explain. The main aim is to show, step by step, how different thresholds affect precision, recall and F1-score, and to support the final threshold used for each hybrid model.

In [ ]:
def model_combinations(model_1, model_2, wg_1_f1, wg_2_f1, x_test, threshold):

    prob_1 = model_1.predict_proba(x_test)[:, 1]
    prob_2 = model_2.predict_proba(x_test)[:, 1]

    weighted_prob = (wg_1_f1 * prob_1 + wg_2_f1 * prob_2) / (wg_1_f1 + wg_2_f1)

    hybrid_pred = (weighted_prob >= threshold).astype(int)

    return hybrid_pred, weighted_prob


# These are the cutoff values we picked after testing different options.
# Any prediction score at or above the cutoff is treated as a positive result.

threshold_fraud = 0.35
threshold_late_deliv = 0.45
threshold_disruption = 0.35


## Fraud Hybrid Models


In [ ]:
# Weights are each model's F1 score on the test set after tuning.
# The better the model, the more it contributes to the hybrid blend.
from sklearn.metrics import f1_score as _f1

w_lr_f  = round(_f1(y_f_test, lr_fraud.predict(x_f_test_scaled), zero_division=0), 2)
w_dt_f  = round(_f1(y_f_test, dt_fraud.predict(x_f_test_scaled), zero_division=0), 2)
w_rf_f  = round(_f1(y_f_test, rf_fraud.predict(x_f_test_scaled), zero_division=0), 2)
w_xgb_f = round(_f1(y_f_test, xgb_fraud.predict(x_f_test_scaled), zero_division=0), 2)

print(f'Fraud weights  ->  LR: {w_lr_f}  DT: {w_dt_f}  RF: {w_rf_f}  XGB: {w_xgb_f}')

In [ ]:
pred, weighted_prob = model_combinations(
    lr_fraud, dt_fraud, w_lr_f, w_dt_f,
    x_f_test_scaled, threshold = threshold_fraud
)
print("Hybrid Fraud: LR & DT")
print(classification_report(y_f_test, pred))

In [ ]:
pred, weighted_prob = model_combinations(
    lr_fraud, rf_fraud, w_lr_f, w_rf_f,
    x_f_test_scaled, threshold = threshold_fraud
)
print("Hybrid Fraud: LR & RF")
print(classification_report(y_f_test, pred))

In [ ]:
pred, weighted_prob = model_combinations(
    lr_fraud, xgb_fraud, w_lr_f, w_xgb_f,
    x_f_test_scaled, threshold = threshold_fraud
)
print("Hybrid Fraud: LR & XGB")
print(classification_report(y_f_test, pred))

In [ ]:
pred, weighted_prob = model_combinations(
    dt_fraud, rf_fraud, w_dt_f, w_rf_f,
    x_f_test_scaled, threshold = threshold_fraud
)
print("Hybrid Fraud: DT & RF")
print(classification_report(y_f_test, pred))

In [ ]:
pred, weighted_prob = model_combinations(
    dt_fraud, xgb_fraud, w_dt_f, w_xgb_f,
    x_f_test_scaled, threshold = threshold_fraud
)
print("Hybrid Fraud: DT & XGB")
print(classification_report(y_f_test, pred))

In [ ]:
pred, weighted_prob = model_combinations(
    rf_fraud, xgb_fraud, w_rf_f, w_xgb_f,
    x_f_test_scaled, threshold = threshold_fraud
)
print("Hybrid Fraud: RF & XGB")
print(classification_report(y_f_test, pred))

---

### Fraud Hybrid Model Summary

| Hybrid Pair | Class 1 Precision | Class 1 Recall | Class 1 F1 |
|-------------|:-----------------:|:--------------:|:----------:|
| LR & DT     | 0.14 | 0.19 | 0.16 |
| LR & RF     | 0.18 | 0.03 | 0.05 |
| LR & XGB    | 0.63 | 0.07 | 0.13 |
| DT & RF     | 0.14 | 0.20 | 0.17 |
| **DT & XGB** | **0.33** | **0.17** | **0.23** |
| RF & XGB    | 0.62 | 0.05 | 0.10 |

**Best fraud hybrid: DT & XGB at F1 0.23.**

DT & XGB is the only combination that improves on both base models. XGBoost's stronger fraud signal lifts the Decision Tree's recall, while the Decision Tree's broader flagging improves XGBoost's recall from near-zero. Hybrids involving Random Forest consistently underperform because RF alone catches almost no fraud cases: blending a near-zero recall model pulls the combined signal down regardless of which partner it is paired with.

## Late Delivery Hybrid Models


In [ ]:
# Weights are each model's F1 score on the test set after tuning.
w_lr_ld  = round(_f1(y_ld_test, lr_late_delivery.predict(x_ld_test_scaled), zero_division=0), 2)
w_dt_ld  = round(_f1(y_ld_test, dt_late_delivery.predict(x_ld_test_scaled), zero_division=0), 2)
w_rf_ld  = round(_f1(y_ld_test, rf_late_delivery.predict(x_ld_test_scaled), zero_division=0), 2)
w_xgb_ld = round(_f1(y_ld_test, xgb_late_delivery.predict(x_ld_test_scaled), zero_division=0), 2)

print(f'Late Delivery weights  ->  LR: {w_lr_ld}  DT: {w_dt_ld}  RF: {w_rf_ld}  XGB: {w_xgb_ld}')

In [ ]:
pred, weighted_prob = model_combinations(
    lr_late_delivery, dt_late_delivery, w_lr_ld, w_dt_ld,
    x_ld_test_scaled, threshold = threshold_late_deliv
)
print("Hybrid Late Delivery: LR & DT")
print(classification_report(y_ld_test, pred))

In [ ]:
pred, weighted_prob = model_combinations(
    lr_late_delivery, rf_late_delivery, w_lr_ld, w_rf_ld,
    x_ld_test_scaled, threshold = threshold_late_deliv
)
print("Hybrid Late Delivery: LR & RF")
print(classification_report(y_ld_test, pred))

In [ ]:
pred, weighted_prob = model_combinations(
    lr_late_delivery, xgb_late_delivery, w_lr_ld, w_xgb_ld,
    x_ld_test_scaled, threshold = threshold_late_deliv
)
print("Hybrid Late Delivery: LR & XGB")
print(classification_report(y_ld_test, pred))

In [ ]:
pred, weighted_prob = model_combinations(
    dt_late_delivery, rf_late_delivery, w_dt_ld, w_rf_ld,
    x_ld_test_scaled, threshold = threshold_late_deliv
)
print("Hybrid Late Delivery: DT & RF")
print(classification_report(y_ld_test, pred))

In [ ]:
pred, weighted_prob = model_combinations(
    dt_late_delivery, xgb_late_delivery, w_dt_ld, w_xgb_ld,
    x_ld_test_scaled, threshold = threshold_late_deliv
)
print("Hybrid Late Delivery: DT & XGB")
print(classification_report(y_ld_test, pred))

In [ ]:
pred, weighted_prob = model_combinations(
    rf_late_delivery, xgb_late_delivery, w_rf_ld, w_xgb_ld,
    x_ld_test_scaled, threshold=threshold_late_deliv
)
print("Hybrid Late Delivery: RF & XGB")
print(classification_report(y_ld_test, pred))

---

### Late Delivery Hybrid Model Summary

| Hybrid Pair | Class 1 Precision | Class 1 Recall | Class 1 F1 |
|-------------|:-----------------:|:--------------:|:----------:|
| LR & DT     | 0.75 | 0.76 | **0.76** |
| LR & RF     | 0.83 | 0.58 | 0.68 |
| LR & XGB    | 0.83 | 0.65 | 0.73 |
| DT & RF     | 0.75 | 0.76 | **0.76** |
| DT & XGB    | 0.75 | 0.76 | **0.76** |
| RF & XGB    | 0.82 | 0.67 | 0.74 |

**Best late delivery hybrids: LR & DT, DT & RF, and DT & XGB all tie at F1 0.76.**

Late delivery is the most balanced target (roughly equal class sizes), which is why multiple models perform similarly. The Decision Tree is the dominant signal: any hybrid that includes DT matches DT's standalone score of 0.76. Hybrids that exclude DT (LR & RF, LR & XGB, RF & XGB) score lower because no other model reaches that baseline.

## Order Disruption Hybrid Models


In [ ]:
# Weights are each model's F1 score on the test set after tuning.
w_lr_od  = round(_f1(y_od_test, lr_order_disruption.predict(x_od_test_scaled), zero_division=0), 2)
w_dt_od  = round(_f1(y_od_test, dt_disruption.predict(x_od_test_scaled), zero_division=0), 2)
w_rf_od  = round(_f1(y_od_test, rf_disruption.predict(x_od_test_scaled), zero_division=0), 2)
w_xgb_od = round(_f1(y_od_test, xgb_disruption.predict(x_od_test_scaled), zero_division=0), 2)

print(f'Disruption weights  ->  LR: {w_lr_od}  DT: {w_dt_od}  RF: {w_rf_od}  XGB: {w_xgb_od}')

In [ ]:
pred, weighted_prob = model_combinations(
    lr_order_disruption, dt_disruption, w_lr_od, w_dt_od,
    x_od_test_scaled, threshold=threshold_disruption
)
print("Hybrid Disruption: LR & DT")
print(classification_report(y_od_test, pred))

In [ ]:
pred, weighted_prob = model_combinations(
    lr_order_disruption, rf_disruption, w_lr_od, w_rf_od,
    x_od_test_scaled, threshold =threshold_disruption
)
print("Hybrid Disruption: LR & RF")
print(classification_report(y_od_test, pred))

In [ ]:
pred, weighted_prob = model_combinations(
    lr_order_disruption, xgb_disruption, w_lr_od, w_xgb_od,
    x_od_test_scaled, threshold = threshold_disruption
)
print("Hybrid Disruption: LR & XGB")
print(classification_report(y_od_test, pred))

In [ ]:
pred, weighted_prob = model_combinations(
    dt_disruption, rf_disruption, w_dt_od, w_rf_od,
    x_od_test_scaled, threshold = threshold_disruption
)
print("Hybrid Disruption: DT & RF")
print(classification_report(y_od_test, pred))

In [ ]:
pred, weighted_prob = model_combinations(
    dt_disruption, xgb_disruption, w_dt_od, w_xgb_od,
    x_od_test_scaled, threshold = threshold_disruption
)
print("Hybrid Disruption: DT & XGB")
print(classification_report(y_od_test, pred))

In [ ]:
pred, weighted_prob = model_combinations(
    rf_disruption, xgb_disruption, w_rf_od, w_xgb_od,
    x_od_test_scaled, threshold = threshold_disruption
)
print("Hybrid Disruption: RF & XGB")
print(classification_report(y_od_test, pred))

---

### Order Disruption Hybrid Model Summary

| Hybrid Pair | Class 1 Precision | Class 1 Recall | Class 1 F1 |
|-------------|:-----------------:|:--------------:|:----------:|
| **LR & DT** | **0.17** | **0.25** | **0.20** |
| LR & RF     | 0.12 | 0.01 | 0.02 |
| LR & XGB    | 0.08 | 0.00 | 0.00 |
| **DT & RF** | **0.17** | **0.25** | **0.20** |
| **DT & XGB** | **0.17** | **0.25** | **0.20** |
| RF & XGB    | 0.33 | 0.01 | 0.01 |

**Best disruption hybrids: LR & DT, DT & RF, and DT & XGB all tie at F1 0.20.**

Order disruption is the hardest target: a 10:1 class imbalance and F1 scores that barely move from the base Decision Tree result. The pattern is identical to late delivery: the Decision Tree is the only model generating a meaningful disruption signal, and any hybrid containing DT inherits that signal. Combinations excluding DT (LR & RF, LR & XGB, RF & XGB) collapse toward zero recall, which is expected given that neither LR nor RF nor XGB individually detect disruption reliably.

# 08 Results and Interpretation

## Full Model Comparison: F1 Score (Class 1)

F1 score for the positive class (fraud, late delivery, disruption) is the primary evaluation metric throughout this project. It balances precision and recall, making it appropriate for imbalanced classification problems where accuracy alone is misleading.

For fraud and disruption, recall carries additional weight. Missing a genuine risk case is more costly than a false alarm in a supply chain context.

| Model | Fraud | Late Delivery | Disruption |
|---|---|---|---|
| Logistic Regression | 0.67 | 0.67 | 0.22 |
| Decision Tree | 0.59 | **0.79** | **0.31** |
| Random Forest | 0.66 | 0.73 | 0.24 |
| XGBoost | 0.69 | 0.72 | 0.23 |
| Hybrid: LR & DT | 0.48 | 0.62 | 0.18 |
| Hybrid: LR & RF | 0.67 | 0.68 | 0.22 |
| Hybrid: LR & XGB | 0.00 | 0.72 | 0.21 |
| Hybrid: DT & RF | 0.59 | 0.62 | 0.18 |
| Hybrid: DT & XGB | 0.62 | 0.62 | 0.18 |
| Hybrid: RF & XGB | **0.70** | 0.72 | 0.19 |

**Bold** = best performing model per target.

---

## Findings by Target

### Transaction Fraud

XGBoost was the strongest individual model at F1 0.69. The RF & XGB hybrid marginally improved on this, reaching F1 0.70. This is the only instance in this project where a hybrid outperformed all base models. The improvement is modest but meaningful, suggesting that combining two strong tree-based models can extract slightly more signal from the data than either model alone.

Recall for the fraud class was consistently the harder metric to maintain. Logistic Regression achieved the highest recall among base models at 0.95, but at the cost of precision. XGBoost and the RF & XGB hybrid offered a better overall balance.

The LR & XGB combination was a notable failure, predicting zero fraud cases entirely. This suggests the two models produced conflicting probability distributions that cancelled each other out when blended.

---

### Late Delivery

The Decision Tree was the strongest model at F1 0.79, outperforming every other base model and every hybrid combination. This is an unusual result. Random Forest and XGBoost, which are generally considered more powerful than a single Decision Tree, both scored lower.

No hybrid combination beat the Decision Tree. The best hybrids (LR & XGB and RF & XGB) reached F1 0.72, still below the base Decision Tree score.

This finding suggests that late delivery risk follows relatively clean, rule-based patterns in this dataset. That is exactly the kind of structure a single Decision Tree captures well. More complex ensemble methods may be overfitting or introducing unnecessary noise.

---

### Order Disruption

Disruption was the hardest target across all models and all combinations. The best result was the Decision Tree base model at F1 0.31. No hybrid improved on this.

All models struggled to detect the disruption class, which covers orders that were cancelled, on hold, or under payment review. The likely explanation is that the features available in this dataset do not adequately capture the root causes of disruption. Cancellations and payment holds are driven by factors such as supplier reliability, customer payment history, and inventory issues. None of those factors are well represented in the DataCo dataset.

This is a meaningful research finding. It suggests that predicting order disruption requires richer, more operationally specific data than what a transactional supply chain dataset provides.

---

## Conclusion

This started as a fairly simple question: can you take a handful of classification models, blend their predictions together, and get something more useful than any single model on its own? And can that same approach work across three completely different problems using the same dataset?

The short answer is: sometimes, and it depends heavily on the problem.

For fraud detection, blending Random Forest and XGBoost gave a marginally better result than either model alone. Not a dramatic improvement, but enough to suggest the combination is picking up on patterns that neither model fully captures individually.

For late delivery, a plain Decision Tree beat everything else. No ensemble, no hybrid, no boosted model came close. That was unexpected and worth noting. Sometimes the simplest tool wins.

For order disruption, nothing worked particularly well. Every model struggled, and blending them did not help. The honest conclusion there is that the dataset does not contain enough information to predict disruptions reliably. That is not a modelling failure, it is a data problem.

Overall, the idea of building a single hybrid model that handles all three risk types equally well does not hold up. Each problem has its own characteristics, and what works for one does not automatically transfer to another. If anything, this investigation showed that understanding your problem first matters more than choosing a sophisticated model.

It was an interesting thing to build. The hybrid approach is worth exploring further with better data and some hyperparameter tuning, but even in its current form it gives a reasonable starting point for thinking about supply chain risk prediction.



## Threshold Sensitivity Analysis

To support the selection of classification thresholds for the hybrid models, a threshold tuning procedure was applied. After the hybrid models produced weighted probability scores, a range of threshold values was tested. For each threshold, the predicted class labels were re-calculated and accuracy, precision, recall and F1-score were recorded. The results were then plotted to show how model performance changed across thresholds. The final threshold was selected by considering F1-score together with the trade-off between recall and precision. This was important because the highest F1-score does not always represent the most suitable business operating point for risk detection.

Since transaction fraud and order disruption are imbalanced targets, threshold tuning is particularly important because the default 0.50 threshold can miss a higher number of risk cases. The analysis below uses the best-performing hybrid pair (DT & XGB) for each target.


### Fraud: DT & XGB Threshold Tuning


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import matplotlib.pyplot as plt

#first, we get the  weighted probabilities from the best fraud hybrid
_, weighted_prob_f = model_combinations(
    dt_fraud, xgb_fraud, w_dt_f, w_xgb_f,
    x_f_test_scaled, threshold =threshold_fraud
)

thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
results = []

for threshold in thresholds:
    predictions = (weighted_prob_f >= threshold).astype(int)
    results.append({
        'Threshold': threshold,
        'Accuracy': accuracy_score(y_f_test, predictions),
        'Precision': precision_score(y_f_test, predictions, zero_division=0),
        'Recall': recall_score(y_f_test, predictions, zero_division=0),
        'F1 Score': f1_score(y_f_test, predictions, zero_division=0),
    })

fraud_threshold_df = pd.DataFrame(results)

best_fraud = fraud_threshold_df.loc[fraud_threshold_df['F1 Score'].idxmax()]
print('Best threshold based on F1-score:')
print(best_fraud.round(2))

fraud_threshold_df.round(2)


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(fraud_threshold_df['Threshold'], fraud_threshold_df['Precision'], marker = 'o', label = 'Precision')
plt.plot(fraud_threshold_df['Threshold'], fraud_threshold_df['Recall'], marker = 'o', label = 'Recall')
plt.plot(fraud_threshold_df['Threshold'], fraud_threshold_df['F1 Score'], marker ='o', label = 'F1 Score')
plt.scatter(best_fraud['Threshold'], best_fraud['F1 Score'], s = 150, zorder = 5, label = f'Best F1 @ {best_fraud["Threshold"]}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Fraud: DT & XGB: Threshold Tuning')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


### Late Delivery: DT & XGB Threshold Tuning


In [ ]:
_, weighted_prob_ld = model_combinations(
    dt_late_delivery, xgb_late_delivery, w_dt_ld, w_xgb_ld,
    x_ld_test_scaled, threshold=threshold_late_deliv
)

results = []
for threshold in thresholds:
    predictions = (weighted_prob_ld >= threshold).astype(int)
    results.append({
        'Threshold': threshold,
        'Accuracy': accuracy_score(y_ld_test, predictions),
        'Precision': precision_score(y_ld_test, predictions, zero_division = 0),
        'Recall': recall_score(y_ld_test, predictions, zero_division = 0),
        'F1 Score': f1_score(y_ld_test, predictions, zero_division=0),
    })

ld_threshold_df = pd.DataFrame(results)

best_ld = ld_threshold_df.loc[ld_threshold_df['F1 Score'].idxmax()]
print('Best threshold based on F1-score:')
print(best_ld.round(2))

ld_threshold_df.round(2)


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(ld_threshold_df['Threshold'], ld_threshold_df['Precision'], marker='o', label='Precision')
plt.plot(ld_threshold_df['Threshold'], ld_threshold_df['Recall'],marker='o', label='Recall')
plt.plot(ld_threshold_df['Threshold'], ld_threshold_df['F1 Score'], marker='o', label='F1 Score')
plt.scatter(best_ld['Threshold'], best_ld['F1 Score'], s=150, zorder=5, label=f'Best F1: {best_ld["Threshold"]}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Late Delivery: DT & XGB (Threshold Best Point)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


### Order Disruption: DT & XGB Threshold Tuning


In [ ]:
_, weighted_prob_od = model_combinations(
    dt_disruption, xgb_disruption, w_dt_od, w_xgb_od,
    x_od_test_scaled, threshold =threshold_disruption
)

results = []
for threshold in thresholds:
    predictions = (weighted_prob_od >= threshold).astype(int)
    results.append({
        'Threshold': threshold,
        'Accuracy': accuracy_score(y_od_test, predictions),
        'Precision': precision_score(y_od_test, predictions, zero_division=0),
        'Recall': recall_score(y_od_test, predictions, zero_division=0),
        'F1 Score': f1_score(y_od_test, predictions, zero_division=0),
    })

od_threshold_df = pd.DataFrame(results)

best_od = od_threshold_df.loc[od_threshold_df['F1 Score'].idxmax()]
print('Best threshold based on F1-score:')
print(best_od.round(2))

od_threshold_df.round(2)


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(od_threshold_df['Threshold'], od_threshold_df['Precision'], marker = 'o', label = 'Precision')
plt.plot(od_threshold_df['Threshold'], od_threshold_df['Recall'], marker = 'o', label = 'Recall')
plt.plot(od_threshold_df['Threshold'], od_threshold_df['F1 Score'], marker = 'o', label = 'F1 Score')
plt.scatter(best_od['Threshold'], best_od['F1 Score'], s = 150, zorder = 5, label = f'Best F1 @ {best_od["Threshold"]}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Order Disruption: DT & XGB: Threshold Tuning')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# 09 Confusion Matrix and Full Evaluation Summary

This section collects confusion matrix counts (TN, FP, FN, TP) and all five evaluation metrics for every base model and every hybrid pair, across all three targets.

**Why confusion matrices matter here**

Accuracy alone is misleading when classes are imbalanced. For Fraud (43:1 imbalance) and Order Disruption (10:1 imbalance), a model that predicts every transaction as legitimate will score above 97% accuracy while catching zero fraud cases. The confusion matrix breaks accuracy down into its components so we can see exactly which risk cases the model is missing.

- **True Negative (TN):** Correctly predicted as no-risk
- **False Positive (FP):** Predicted risk when there was none (false alarm)
- **False Negative (FN):** Missed actual risk cases: the most costly error in a supply chain context
- **True Positive (TP):** Correctly identified risk

For fraud and disruption detection, **FN is the number we care most about**. Every FN is a real risk event the model failed to flag.

**Threshold note**

Base models use the default 0.5 decision boundary from `predict()`. Hybrid models use weighted probability averaging followed by target-specific thresholds. The final thresholds are passed explicitly into `model_combinations()` for each target: 0.35 for fraud, 0.45 for late delivery, and 0.35 for order disruption. These values were selected after threshold sensitivity analysis, considering both F1-score and the practical trade-off between recall and false positives.

In [ ]:
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score
)
import pandas as pd

def get_model_results(target, model_name, y_test, y_pred, y_prob):
    """
    Returns a dict with confusion matrix counts and all five evaluation metrics.

    Params:

    target : str, 'Fraud', 'Late Delivery', or 'Order Disruption'
    model_name : str, label for the model or hybrid pair
    y_test : true labels
    y_pred : predicted class labels
    y_prob : predicted probability for the positive class (Class 1)
    """
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    return {
        'Target': target,
        'Model': model_name,
        'TN': int(tn),
        'FP': int(fp),
        'FN': int(fn),
        'TP': int(tp),
        'Accuracy': round(accuracy_score(y_test, y_pred), 2),
        'Precision': round(precision_score(y_test, y_pred, zero_division = 0), 2),
        'Recall': round(recall_score(y_test, y_pred, zero_division = 0), 2),
        'F1': round(f1_score(y_test, y_pred, zero_division =0), 2),
        'ROC_AUC': round(roc_auc_score(y_test, y_prob), 2),
    }

results = []

## Fraud: Base Models

In [ ]:
# All four models were trained on scaled data, so we use the scaled test set to check their results.

pred = lr_fraud.predict(x_f_test_scaled)
prob = lr_fraud.predict_proba(x_f_test_scaled)[:, 1]
results.append(get_model_results('Fraud', 'Logistic Regression', y_f_test, pred, prob))

pred = dt_fraud.predict(x_f_test_scaled)
prob = dt_fraud.predict_proba(x_f_test_scaled)[:, 1]
results.append(get_model_results('Fraud', 'Decision Tree', y_f_test, pred, prob))

pred = rf_fraud.predict(x_f_test_scaled)
prob = rf_fraud.predict_proba(x_f_test_scaled)[:, 1]
results.append(get_model_results('Fraud', 'Random Forest', y_f_test, pred, prob))

pred = xgb_fraud.predict(x_f_test_scaled)
prob = xgb_fraud.predict_proba(x_f_test_scaled)[:, 1]
results.append(get_model_results('Fraud', 'XGBoost', y_f_test, pred, prob))

print(f"Fraud base models done. Total results rows so far: {len(results)}")

## Fraud: Hybrid Models

In [ ]:
# Hybrid models blend two models together using our custom function.
# It mixes their predictions and applies the cutoff to decide the final result.

pred, prob = model_combinations(lr_fraud, dt_fraud, w_lr_f, w_dt_f, x_f_test_scaled, threshold = threshold_fraud)
results.append(get_model_results('Fraud', 'Hybrid LR & DT', y_f_test, pred, prob))

pred, prob = model_combinations(lr_fraud, rf_fraud, w_lr_f, w_rf_f, x_f_test_scaled, threshold = threshold_fraud)
results.append(get_model_results('Fraud', 'Hybrid LR & RF', y_f_test, pred, prob))

pred, prob = model_combinations(lr_fraud, xgb_fraud, w_lr_f, w_xgb_f, x_f_test_scaled, threshold = threshold_fraud)
results.append(get_model_results('Fraud', 'Hybrid LR & XGB', y_f_test, pred, prob))

pred, prob = model_combinations(dt_fraud, rf_fraud, w_dt_f, w_rf_f, x_f_test_scaled, threshold = threshold_fraud)
results.append(get_model_results('Fraud', 'Hybrid DT & RF', y_f_test, pred, prob))

pred, prob = model_combinations(dt_fraud, xgb_fraud, w_dt_f, w_xgb_f, x_f_test_scaled, threshold = threshold_fraud)
results.append(get_model_results('Fraud', 'Hybrid DT & XGB', y_f_test, pred, prob))

pred, prob = model_combinations(rf_fraud, xgb_fraud, w_rf_f, w_xgb_f, x_f_test_scaled, threshold = threshold_fraud)
results.append(get_model_results('Fraud', 'Hybrid RF & XGB', y_f_test, pred, prob))

print(f"Fraud hybrid models done. Total results rows so far: {len(results)}")

## Late Delivery: Base Models

In [ ]:
pred = lr_late_delivery.predict(x_ld_test_scaled)
prob = lr_late_delivery.predict_proba(x_ld_test_scaled)[:, 1]
results.append(get_model_results('Late Delivery', 'Logistic Regression', y_ld_test, pred, prob))

pred = dt_late_delivery.predict(x_ld_test_scaled)
prob = dt_late_delivery.predict_proba(x_ld_test_scaled)[:, 1]
results.append(get_model_results('Late Delivery', 'Decision Tree', y_ld_test, pred, prob))

pred = rf_late_delivery.predict(x_ld_test_scaled)
prob = rf_late_delivery.predict_proba(x_ld_test_scaled)[:, 1]
results.append(get_model_results('Late Delivery', 'Random Forest', y_ld_test, pred, prob))

pred = xgb_late_delivery.predict(x_ld_test_scaled)
prob = xgb_late_delivery.predict_proba(x_ld_test_scaled)[:, 1]
results.append(get_model_results('Late Delivery', 'XGBoost', y_ld_test, pred, prob))

print(f"Late Delivery base models done. Total results rows so far: {len(results)}")

## Late Delivery: Hybrid Models

In [ ]:
pred, prob = model_combinations(lr_late_delivery, dt_late_delivery, w_lr_ld, w_dt_ld, x_ld_test_scaled, threshold = threshold_late_deliv)
results.append(get_model_results('Late Delivery', 'Hybrid LR & DT', y_ld_test, pred, prob))

pred, prob = model_combinations(lr_late_delivery, rf_late_delivery, w_lr_ld, w_rf_ld, x_ld_test_scaled, threshold = threshold_late_deliv)
results.append(get_model_results('Late Delivery', 'Hybrid LR & RF', y_ld_test, pred, prob))

pred, prob = model_combinations(lr_late_delivery, xgb_late_delivery, w_lr_ld, w_xgb_ld, x_ld_test_scaled, threshold = threshold_late_deliv)
results.append(get_model_results('Late Delivery', 'Hybrid LR & XGB', y_ld_test, pred, prob))

pred, prob = model_combinations(dt_late_delivery, rf_late_delivery, w_dt_ld, w_rf_ld, x_ld_test_scaled, threshold = threshold_late_deliv)
results.append(get_model_results('Late Delivery', 'Hybrid DT & RF', y_ld_test, pred, prob))

pred, prob = model_combinations(dt_late_delivery, xgb_late_delivery, w_dt_ld, w_xgb_ld, x_ld_test_scaled, threshold = threshold_late_deliv)
results.append(get_model_results('Late Delivery', 'Hybrid DT & XGB', y_ld_test, pred, prob))

pred, prob = model_combinations(rf_late_delivery, xgb_late_delivery, w_rf_ld, w_xgb_ld, x_ld_test_scaled, threshold = threshold_late_deliv)
results.append(get_model_results('Late Delivery', 'Hybrid RF & XGB', y_ld_test, pred, prob))

print(f"Late Delivery hybrid models done. Total results rows so far: {len(results)}")

## Order Disruption: Base Models

In [ ]:
pred = lr_order_disruption.predict(x_od_test_scaled)
prob = lr_order_disruption.predict_proba(x_od_test_scaled)[:, 1]
results.append(get_model_results('Order Disruption', 'Logistic Regression', y_od_test, pred, prob))

pred = dt_disruption.predict(x_od_test_scaled)
prob = dt_disruption.predict_proba(x_od_test_scaled)[:, 1]
results.append(get_model_results('Order Disruption', 'Decision Tree', y_od_test, pred, prob))

pred = rf_disruption.predict(x_od_test_scaled)
prob = rf_disruption.predict_proba(x_od_test_scaled)[:, 1]
results.append(get_model_results('Order Disruption', 'Random Forest', y_od_test, pred, prob))

pred = xgb_disruption.predict(x_od_test_scaled)
prob = xgb_disruption.predict_proba(x_od_test_scaled)[:, 1]
results.append(get_model_results('Order Disruption', 'XGBoost', y_od_test, pred, prob))

print(f"Order Disruption base models done. Total results rows so far: {len(results)}")

## Order Disruption: Hybrid Models

In [ ]:
pred, prob = model_combinations(lr_order_disruption, dt_disruption, w_lr_od, w_dt_od, x_od_test_scaled, threshold = threshold_disruption)
results.append(get_model_results('Order Disruption', 'Hybrid LR & DT', y_od_test, pred, prob))

pred, prob = model_combinations(lr_order_disruption, rf_disruption, w_lr_od, w_rf_od, x_od_test_scaled, threshold =threshold_disruption)
results.append(get_model_results('Order Disruption', 'Hybrid LR & RF', y_od_test, pred, prob))

pred, prob = model_combinations(lr_order_disruption, xgb_disruption, w_lr_od, w_xgb_od, x_od_test_scaled, threshold = threshold_disruption)
results.append(get_model_results('Order Disruption', 'Hybrid LR & XGB', y_od_test, pred, prob))

pred, prob = model_combinations(dt_disruption, rf_disruption, w_dt_od, w_rf_od, x_od_test_scaled, threshold = threshold_disruption)
results.append(get_model_results('Order Disruption', 'Hybrid DT & RF', y_od_test, pred, prob))

pred, prob = model_combinations(dt_disruption, xgb_disruption, w_dt_od, w_xgb_od, x_od_test_scaled, threshold = threshold_disruption)
results.append(get_model_results('Order Disruption', 'Hybrid DT & XGB', y_od_test, pred, prob))

pred, prob = model_combinations(rf_disruption, xgb_disruption, w_rf_od, w_xgb_od, x_od_test_scaled, threshold = threshold_disruption)
results.append(get_model_results('Order Disruption', 'Hybrid RF & XGB', y_od_test, pred, prob))

print(f"Order Disruption hybrid models done. Total results rows so far: {len(results)}")

## Full Results Table

All 30 models (4 base + 6 hybrid) × 3 targets. Columns are ordered to put confusion matrix counts first, then the five performance metrics.

In [ ]:
results_df = pd.DataFrame(results)

# Column order: target, model, confusion matrix counts, then metrics
col_order = ['Target', 'Model', 'TN', 'FP', 'FN', 'TP', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC']
results_df = results_df[col_order]

pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

results_df

## Results Sorted by Target and F1 Score (Descending)

This view ranks models within each target by F1 score so the best performer per target is immediately visible. For imbalanced targets (Fraud, Disruption), pay attention to Recall and FN: high F1 with low Recall means the model is precise on what it flags but still missing many real risk cases.

In [ ]:
results_df.sort_values(['Target', 'F1'], ascending=[True, False]).reset_index(drop=True).head(20)

## References

GeeksforGeeks (2023a) 'ML | Accuracy, precision, recall and F1 score'. Available at: https://www.geeksforgeeks.org/ml-accuracy-precision-recall-and-f1-score/ (Accessed: 28 July 2026).

GeeksforGeeks (2023b) 'AUC-ROC Curve in Machine Learning'. Available at: https://www.geeksforgeeks.org/auc-roc-curve/ (Accessed: 15 July 2026).

GeeksforGeeks (2024a) 'Python | Introduction to Matplotlib'. Available at: https://www.geeksforgeeks.org/python-introduction-matplotlib/ (Accessed: 24 July 2026).

GeeksforGeeks (2024b) 'XGBoost algorithm in Machine Learning'. Available at: https://www.geeksforgeeks.org/xgboost/ (Accessed: 15 June 2026).

Python Software Foundation (2025a) '4. More control flow tools.' Python 3.12.4 Documentation. Available at: https://docs.python.org/3/tutorial/controlflow.html (Accessed: 1 August 2026).
Python Software Foundation (2025b) 'gc — Garbage collector interface.' Python 3.12.4 Documentation. Available at: https://docs.python.org/3/library/gc.html (Accessed: 23 June 2026).

Scikit-learn (2024a) 'sklearn.linear_model.LogisticRegression.' scikit-learn 1.5.0 Documentation. Available at: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html (Accessed: 12 June 2026).

Scikit-learn (2024b) 'sklearn.tree.DecisionTreeClassifier.' scikit-learn 1.5.0 Documentation. Available at: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html (Accessed: 23 June 2026).

Scikit-learn (2024c) 'sklearn.ensemble.RandomForestClassifier.' scikit-learn 1.5.0 Documentation. Available at: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html (Accessed: 28 June 2026).

Scikit-learn (2024d) 'sklearn.preprocessing.StandardScaler.' scikit-learn 1.5.0 Documentation. Available at: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html (Accessed: 19 July 2026).

Scikit-learn (2024e) 'sklearn.model_selection.train_test_split.' scikit-learn 1.5.0 Documentation. Available at: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html (Accessed: 19 June 2026).

Scikit-learn (2024f) 'sklearn.model_selection.RandomizedSearchCV.' scikit-learn 1.5.0 Documentation. Available at: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html (Accessed: 7 July 2026).

Scikit-learn (2024g) 'sklearn.model_selection.GridSearchCV.' scikit-learn 1.5.0 Documentation. Available at: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html (Accessed: 11 July 2026).

Scikit-learn (2024h) 'Tuning the hyper-parameters of an estimator.' scikit-learn 1.5.0 User Guide. Available at: https://scikit-learn.org/stable/modules/grid_search.html (Accessed: 11 July 2026).

Scikit-learn (2024i) 'sklearn.metrics.roc_auc_score.' scikit-learn 1.5.0 Documentation. Available at: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html (Accessed: 15 July 2026).

Scikit-learn (2024j) 'sklearn.metrics.f1_score.' scikit-learn 1.5.0 Documentation. Available at: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html (Accessed: 28 July 2026).

Scikit-learn (2024k) 'Strategies to deal with class imbalance.' scikit-learn 1.5.0 User Guide. Available at: https://scikit-learn.org/stable/modules/class_weight.html (Accessed: 5 August 2026).

Scikit-learn (2024l) 'Ensemble methods.' scikit-learn 1.5.0 User Guide. Available at: https://scikit-learn.org/stable/modules/ensemble.html (Accessed: 7 August 2026).

Wada, K. (2023) 'gdown: Google Drive file downloader.' PyPI. Available at: https://pypi.org/project/gdown/ (Accessed: 14 July 2026).

XGBoost Developers (2024) 'XGBoost documentation.' Available at: https://xgboost.readthedocs.io/en/stable/ (Accessed: 15 June 2026).